[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-02.ipynb)

# 2주차 실습: 학습 루프와 텐서 연산

**목표.** 손실을 계산하고 역전파로 가중치를 갱신하는 **학습 루프 한 바퀴**를 직접 돌려 손실이 내려가는 것을 확인한다. 표기(방언 vs 표준어)가 토크나이저에서 다른 비용으로 이어지는 것도 실측한다.

이 노트북은 이후 모든 주차 실습이 재사용하는 **기본 학습 루프**다. 처음부터 끝까지 한 번 돌려서 감을 잡는 것이 이번 주 목표다.


## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. Colab 무료 런타임이면 이미 설치돼 있을 수 있지만, 안전하게 한 번 실행한다.


In [1]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers datasets


## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.


### 1-1. 텐서와 자동미분

2주차 온라인에서 본 **자동미분**의 동작을 숫자로 확인한다. `requires_grad=True` 텐서에 연산을 걸면 `backward()` 호출 시 그래디언트가 채워진다.


In [1]:
import torch

# x 에 대해 loss 가 미분 가능한지 확인한다
x = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

loss = (w * x + b - 10) ** 2   # w*x + b 가 10 에 가까워지도록 하는 손실
loss.backward()

print(f"loss = {loss.item():.4f}")
print(f"d(loss)/dw = {w.grad.item():.4f}")
print(f"d(loss)/dx = {x.grad.item():.4f}")
print(f"d(loss)/db = {b.grad.item():.4f}")


loss = 9.0000
d(loss)/dw = -18.0000
d(loss)/dx = -12.0000
d(loss)/db = -6.0000


### 1-2. 학습 루프 한 바퀴

이제 **순전파 - 손실 - 역전파 - 갱신** 순서를 한 바퀴 돌린다. 가중치가 한 번 갱신된 뒤 손실이 어떻게 변하는지 본다.


In [2]:
import torch

# 데이터: x 가 주어졌을 때 3*x + 1 에 가까운 값을 내는 선형 모델을 맞춘다
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[4.0], [7.0], [10.0], [13.0]])  # 정답: 3*x + 1

w = torch.tensor([[0.5]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)

def predict(x):
    return x @ w + b

def mean_squared_error(pred, target):
    return ((pred - target) ** 2).mean()

lr = 0.01

for step in range(200):
    pred = predict(x)
    loss = mean_squared_error(pred, y)
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if step % 40 == 0:
        print(f"step {step:3d}  loss = {loss.item():.6f}")

print(f"학습 후 w = {w.item():.3f}, b = {b.item():.3f}  (정답: w=3, b=1)")


step   0  loss = 60.375000
step  40  loss = 0.002392
step  80  loss = 0.001861
step 120  loss = 0.001464
step 160  loss = 0.001152
학습 후 w = 3.025, b = 0.926  (정답: w=3, b=1)


### 1-3. 한국어 텍스트 분류 모델 로딩

2주차 온라인에서 본 **사전학습 모델 로딩**이다. 작은 한국어 분류 모델을 불러온다. 3주차에서 이 모델에 어댑터(LoRA)를 붙일 때 같은 코드를 쓴다.


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"모델 전체 파라미터 수: {model.num_parameters():,}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your do

모델 전체 파라미터 수: 112,922,882


## 2. 한 지점만 바꿔 보기

아래 셀의 `# TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 바꾸기 전 결과를 먼저 확인해 두면 무엇이 달라졌는지 비교할 수 있습니다.


In [5]:
import torch

# x 가 주어졌을 때 3*x + 1 에 가까운 값을 내는 모델 (1-2 와 동일)
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[4.0], [7.0], [10.0], [13.0]])

w = torch.tensor([[0.5]], requires_grad=True)
b = torch.tensor([[0.0]], requires_grad=True)

def predict(x):
    return x @ w + b

def mean_squared_error(pred, target):
    return ((pred - target) ** 2).mean()

# TODO: 손실 함수를 mean_squared_error 에서 mean_absolute_error 로 바꿔 보세요
#       아래 def 한 줄을 바꾸면 됩니다.
def mean_absolute_error(pred, target):
    return (pred - target).abs().mean()

loss_fn = mean_absolute_error

lr = 0.01
for step in range(200):
    pred = predict(x)
    loss = loss_fn(pred, y)
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if step % 40 == 0:
        print(f"step {step:3d}  loss = {loss.item():.6f}")
print(f"학습 후 w = {w.item():.3f}, b = {b.item():.3f}  (정답: w=3, b=1)")

step   0  loss = 7.250000
step  40  loss = 4.350002
step  80  loss = 1.449999
step 120  loss = 0.000005
step 160  loss = 0.000005
학습 후 w = 3.000, b = 1.000  (정답: w=3, b=1)


## 3. 실무: 방언·표준어 표기와 토크나이저 비용

[실무] 같은 뜻을 담는 데 드는 토큰 비용이 **표기에 따라** 달라지는지 실측한다. 사전학습 토크나이저는 표준어 말뭉치로 학습됐으므로, 방언 표기(제주어)는 같은 뜻의 표준어보다 **토큰 수가 더 많아질** 가능성이 크다.


### 3-1. 먼저 샘플로 감 잡기

데이터를 올리기 전에, 방언과 표준어가 토크나이저에서 어떻게 쪼개지는지 작은 예로 확인한다.


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("monologg/koelectra-base-v3-discriminator")

pairs = [
    ("혼저 옵서예.", "어서 오세요."),        # 방언 / 표준어
    ("마씀", "말씀"),
    ("고맙수다", "고맙습니다"),
]

for dialect, standard in pairs:
    d_tok = tokenizer.encode(dialect, add_special_tokens=False)
    s_tok = tokenizer.encode(standard, add_special_tokens=False)
    print(f"방언  [{dialect}] -> {len(d_tok)} 토큰 : {tokenizer.convert_ids_to_tokens(d_tok)}")
    print(f"표준  [{standard}] -> {len(s_tok)} 토큰 : {tokenizer.convert_ids_to_tokens(s_tok)}")
    print()


방언  [혼저 옵서예.] -> 6 토큰 : ['혼', '##저', '옵', '##서', '##예', '.']
표준  [어서 오세요.] -> 4 토큰 : ['어서', '오세', '##요', '.']

방언  [마씀] -> 2 토큰 : ['마', '##씀']
표준  [말씀] -> 1 토큰 : ['말씀']

방언  [고맙수다] -> 3 토큰 : ['고맙', '##수', '##다']
표준  [고맙습니다] -> 3 토큰 : ['고맙', '##습', '##니다']



### 3-2. AI Hub 「한국어 방언 발화(제주도)」 데이터 올리기

AI Hub(aihub.or.kr)는 로그인이 필요해 Colab에서 바로 받을 수 없다. 아래 순서로 데이터를 준비한다.

1. [AI Hub](https://www.aihub.or.kr) 로그인 → 「한국어 방언 발화(제주도)」 데이터셋을 내려받는다
2. 이 노트북이 실행 중인 Colab 화면 왼쪽 **폴더(파일)** 아이콘을 누른다
3. 내려받은 파일(예: 방언-표준어 대응 텍스트)을 끌어다 놓는다

데이터가 아직 준비되지 않았다면 **이 셀은 건너뛰고** 3-3의 확인 질문으로 넘어가도 된다. 데이터 형식에 맞게 `data` 리스트를 채우면 그대로 이어서 실행된다.


In [ ]:
# data 를 (방언, 표준어) 튜플 리스트로 채우세요. 준비 안 됐으면 그대로 두고 실행해도 됩니다.
data = [
    ("혼저 옵서예.", "어서 오세요."),
    ("하르방", "할아버지"),
    ("쉰 살", "쉰 살"),
]

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("monologg/koelectra-base-v3-discriminator")

print("방언/표준어별 토큰 수:")
for dialect, standard in data:
    d_n = len(tokenizer.encode(dialect, add_special_tokens=False))
    s_n = len(tokenizer.encode(standard, add_special_tokens=False))
    print(f"  방언 [{dialect}] {d_n:2d} 토큰 | 표준 [{standard}] {s_n:2d} 토큰 | 차이 {d_n - s_n:+d}")


## 4. 확인 질문

1. 2번에서 손실 함수를 바꿨을 때 학습 곡선(손실 감소 속도·마지막 값)은 어떻게 달라졌나요? 왜 그렇게 됐다고 생각하나요?
2. 학습 루프에서 **역전파(`loss.backward()`)** 와 **갱신(`w -= lr*w.grad`)** 이 각각 무엇을 하는지 한 문장씩 설명하세요.
3. 방언 표기가 표준어보다 토큰 수가 늘어난 예가 있었나요? 같은 뜻을 담는 데 왜 토큰 비용이 달라지는지 토크나이저의 학습 방식과 연결지어 설명하세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.


1. 학습 곡선 변화: mean_squared_error (MSE)를 사용했을 때는 초기 손실 값이 크게 시작(약 60.37)하고 초반에 급격하게 줄어들다가 뒤로 갈수록 완만해지는 경향을 보였습니다. 반면, mean_absolute_error (MAE)로 바꾸었을 때는 초기 손실(7.25)이 MSE에 비해 상대적으로 매우 작게 시작하였고, 일정한 속도(선형적)로 손실이 감소하여 최종적으로 오차가 거의 0에 수렴하였습니다.
원인: MSE는 오차를 제곱((pred - target) ** 2)하기 때문에 오차가 클수록 그래디언트(기울기)가 매우 커져 초반 갱신 폭이 큽니다. 반면 MAE는 오차의 절대값만 취하므로 오차 크기와 상관없이 그래디언트의 크기가 일정하여 일정한 속도로 학습이 진행됩니다. 이번 실습의 데이터셋은 이상치(outlier)가 없는 깨끗한 선형 관계를 가지므로 MAE를 사용했을 때 더 안정적이고 정확하게 정답(w=3, b=1)에 완벽하게 수렴할 수 있었습니다.

2.*(여기에 답을 적으세요)*역전파 (loss.backward()): 예측값과 실제 정답 사이의 손실(Loss)을 기준으로, 모델의 각 매개변수(가중치 $w$$w$, 편향 $b$$b$)가 손실에 미친 영향도(기울기/그래디언트)를 체인 룰(연쇄 법칙)을 통해 역방향으로 계산하고 기록하는 과정입니다.
갱신 (w -= lr*w.grad): 역전파 단계에서 계산된 그래디언트(w.grad)에 학습률(Learning Rate, lr)을 곱한 만큼 가중치를 반대 방향으로 이동시켜, 손실(Loss)이 최소가 되는 방향으로 실제 매개변수 값을 업데이트하는 과정입니다.

3.실제 예시: 방언인 [혼저 옵서예.]는 6 토큰(['혼', '##저', '옵', '##서', '##예', '.'])으로 분절된 반면, 표준어인 [어서 오세요.]는 4 토큰(['어서', '오세', '##요', '.'])으로 분절되어 방언의 토큰 수가 더 많았습니다. [마씀](2 토큰) 역시 표준어 [말씀](1 토큰)보다 토큰 수가 늘어났습니다.
토크나이저 학습 방식과의 연결: koelectra 모델의 토크나이저는 대규모 한국어 표준어 웹 문서와 뉴스 등의 말뭉치(Corpus)를 기반으로 학습되었습니다. 토크나이저는 자주 등장하는 단어나 글자 조합을 하나의 토큰(단어 사전)으로 등록해 보관하는데, 표준어는 자주 등장하므로 '말씀', '어서'와 같이 한 번에 묶여 효율적으로 처리됩니다. 반면 자주 쓰이지 않는 방언(제주어 등)은 사전에 등록되어 있지 않아 형태소나 글자 단위(##저, ##서, ##예 등)로 잘게 쪼개지게 되며, 이로 인해 동일한 의미를 전달하더라도 더 많은 토큰 비용(더 많은 토큰 수)이 발생하게 됩니다.

## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-02/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.
